# Gradient Boosting: High Accuracy on Business Tables
### Course template: Machine Learning for Business Insights (Module 5)

**What this notebook does**
1. Loads your CSV, defines a target, and runs quick quality checks
2. Builds a leakage-safe pipeline
3. Compares a baseline, a simple linear benchmark, and a default boosting model
4. Tunes boosting with **RandomizedSearchCV + cross-validation**, including **early stopping**
5. Evaluates in business terms (threshold and expected value for yes/no; error in units for numbers)
6. Explains the model: what matters, and in which direction

**How boosting works (in one analogy):** Picture a team of junior analysts working in sequence.
The first makes rough guesses. The second studies only the first one's *mistakes* and corrects them.
The third corrects what's still wrong, and so on. Hundreds of small corrections add up to a very
accurate forecast. The risk: keep going too long and the team starts "correcting" random noise.
That's overfitting, and tuning plus early stopping is how we prevent it.

**Works for:** churn, risk scoring, lead conversion (classification), and sales, demand, or cost forecasting (regression).
Edit **only the CONFIG cell**, then run top to bottom.

In [ ]:
# ============================================================
# IMPORTS (preinstalled in Colab; needs scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, loguniform

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, RepeatedKFold, RepeatedStratifiedKFold,
    TimeSeriesSplit, cross_validate, cross_val_predict, RandomizedSearchCV, learning_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
RANDOM_STATE = 42
print("Libraries loaded.")

# WHY HistGradientBoosting (not GradientBoostingClassifier)?
# - Much faster (it groups values into bins), and works well from ~1,000 up to millions of rows
# - Handles blank values natively (no imputing numbers needed)
# - Handles categories natively (no one-hot explosion)
# GradientBoostingClassifier/Regressor is the older, slower version. It's fine for
# under ~10,000 rows if you prefer it, but needs imputation and gives the same kind of results.

In [ ]:
# ============================================================
# CONFIG: EDIT THIS CELL
# ============================================================

# --- 1. Task -------------------------------------------------
TASK = "classification"         # "classification" (yes/no) or "regression" (a number)
IS_CLF = TASK == "classification"

# --- 2. Data -------------------------------------------------
CSV_PATH = None                 # e.g. "/content/my_data.csv"; None = demo data

# --- 3. Target -----------------------------------------------
TARGET_COLUMN = "defaulted" if IS_CLF else "weekly_sales"
POSITIVE_VALUE = 1              # classification: the value meaning "YES" (1, "Yes", "churned")

# --- 4. Inputs -----------------------------------------------
NUMERIC_COLUMNS = (["credit_limit", "utilization_pct", "avg_days_late_12m",
                    "years_in_business", "invoices_12m"] if IS_CLF else
                   ["price", "promo", "ad_spend", "store_size_sqft", "temperature"])
CATEGORICAL_COLUMNS = (["industry", "region", "payment_terms"] if IS_CLF else
                       ["store_type", "region"])

# Date column (optional). If set, the notebook:
#   - adds month and week-of-year as inputs
#   - uses a TIME-BASED split (train on the past, test on the most recent period)
# Use this for any forecasting problem. Set to None for non-time data.
DATE_COLUMN = None if IS_CLF else "week_start"

# IDs and LEAKAGE columns (anything not known at prediction time).
# Examples: "collections_status" for default risk, "cancellation_reason" for churn,
# "actual_units_sold" when forecasting sales dollars.
DROP_COLUMNS = ["account_id"] if IS_CLF else ["store_id"]

# --- 5. Metric used to choose the best settings ---------------
# Classification: "average_precision" if YES is rare (<20%), else "roc_auc"
# Regression:     "neg_mean_absolute_error" (typical miss in units; easiest to explain)
MAIN_METRIC = "average_precision" if IS_CLF else "neg_mean_absolute_error"

# --- 6. Business numbers (classification only) ----------------
VALUE_IF_CAUGHT = 2_000     # value of correctly acting on one YES case (e.g. loss avoided)
ACTION_COST = 75            # cost of acting on one flagged case (review, call, tighter terms)
ACTION_SUCCESS_RATE = 0.5   # share of acted-on YES cases where the action actually helps

# --- 7. General -------------------------------------------------
DATA_SIZE_MODE = "auto"         # "auto", "small", "large"
N_SEARCH_ITER = 30              # random combinations to try (40–80 if time allows)
SAMPLE_ROWS_FOR_SEARCH = None   # large data: tune on a sample (e.g. 100_000), refit on all
TEST_SIZE = 0.2
UNIT_LABEL = "$"                # regression: "$", "units", "days", ...
print(f"Task: {TASK} | Target: {TARGET_COLUMN} | Metric: {MAIN_METRIC}")

In [ ]:
# ============================================================
# LOAD DATA. Demo data is SYNTHETIC; don't draw real conclusions.
# ============================================================
def make_demo_clf(n=3_000, seed=RANDOM_STATE):
    """B2B accounts: will this account default in the next 6 months?"""
    rng = np.random.default_rng(seed)
    limit = rng.lognormal(10, 0.7, n).round(-2)
    util = rng.beta(2, 3, n) * 100
    late = rng.gamma(1.2, 4, n).round(1)
    years = rng.integers(0, 40, n)
    inv = rng.poisson(24, n)
    industry = rng.choice(["retail", "construction", "manufacturing", "services", "hospitality"], n)
    region = rng.choice(["midwest", "south", "west", "northeast"], n)
    terms = rng.choice(["net15", "net30", "net60"], n, p=[0.2, 0.6, 0.2])
    # Non-linear + interaction effects: where boosting shines
    logit = (-3.0 + 2.0 * (util > 65) + 0.12 * np.minimum(late, 25)
             + 1.3 * ((years < 3) & (util > 50)) - 0.03 * years
             + 0.6 * (industry == "hospitality") + 0.4 * (industry == "construction")
             + 0.35 * (terms == "net60"))
    y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame({"account_id": range(n), "credit_limit": limit, "utilization_pct": util.round(1),
                       "avg_days_late_12m": late, "years_in_business": years, "invoices_12m": inv,
                       "industry": industry, "region": region, "payment_terms": terms, "defaulted": y})
    df.loc[rng.choice(n, 60, replace=False), "avg_days_late_12m"] = np.nan
    return df

def make_demo_reg(stores=25, weeks=60, seed=RANDOM_STATE):
    """Weekly sales per store, with seasonality, promos, and price effects."""
    rng = np.random.default_rng(seed)
    rows = []
    dates = pd.date_range("2024-01-01", periods=weeks, freq="W-MON")
    for s in range(stores):
        stype = rng.choice(["mall", "street", "outlet"]); reg = rng.choice(["north", "south", "east"])
        size = rng.integers(2_000, 12_000)
        for d in dates:
            price = rng.normal(20, 2); promo = int(rng.random() < 0.25); ads = rng.gamma(2, 400)
            temp = 55 + 25 * np.sin(2 * np.pi * (d.dayofyear - 100) / 365) + rng.normal(0, 5)
            season = 1 + 0.25 * (d.month in (11, 12))
            sales = season * (4_000 + 0.9 * size - 180 * (price - 20)
                              + promo * (2_500 if stype == "mall" else 900)   # interaction
                              + 600 * np.log1p(ads) + 30 * np.clip(temp - 50, 0, None)) + rng.normal(0, 900)
            rows.append([s, d, price, promo, ads, size, temp, stype, reg, sales])
    return pd.DataFrame(rows, columns=["store_id", "week_start", "price", "promo", "ad_spend",
                                       "store_size_sqft", "temperature", "store_type", "region",
                                       "weekly_sales"]).round(2)

if CSV_PATH is None:
    df = make_demo_clf() if IS_CLF else make_demo_reg()
    print("Using DEMO data (synthetic).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK
# ============================================================
NUM_COLS, CAT_COLS = list(NUMERIC_COLUMNS), list(CATEGORICAL_COLUMNS)   # working copies
need = [TARGET_COLUMN] + NUM_COLS + CAT_COLS + ([DATE_COLUMN] if DATE_COLUMN else [])
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Not found in data: {miss}\nYour columns: {list(df.columns)}")
if set(DROP_COLUMNS) & set(NUM_COLS + CAT_COLS):
    raise ValueError("A column is both in DROP_COLUMNS and listed as an input.")

# Clean numbers stored as text ("$1,200", "15%")
for c in NUM_COLS + ([] if IS_CLF else [TARGET_COLUMN]):
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[$,%\s]", "", regex=True), errors="coerce")
        print(f"Converted '{c}' to numbers.")

# ============================================================
# DATE FEATURES (only if DATE_COLUMN is set)
# Month and week-of-year let the model learn seasonality.
# These are known in advance, so they're safe (no leakage).
# ============================================================
if DATE_COLUMN:
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    df = df.dropna(subset=[DATE_COLUMN]).sort_values(DATE_COLUMN).reset_index(drop=True)
    df["month"] = df[DATE_COLUMN].dt.month
    df["week_of_year"] = df[DATE_COLUMN].dt.isocalendar().week.astype(int)
    NUM_COLS += ["month", "week_of_year"]
    print(f"Date range: {df[DATE_COLUMN].min().date()} to {df[DATE_COLUMN].max().date()}")
    # TIP: for serious forecasting, add LAG features (e.g. last week's sales for the same store),
    # computed with groupby().shift(). Never use the current week's actual value as an input.

# Store numeric inputs as decimals (floats); partial dependence charts refuse whole-number columns
df[NUM_COLS] = df[NUM_COLS].astype(float)

# ============================================================
# TARGET → _y
# ============================================================
df = df.dropna(subset=[TARGET_COLUMN])
if IS_CLF:
    print("Target values:", df[TARGET_COLUMN].value_counts().to_dict())
    df["_y"] = (df[TARGET_COLUMN].astype(str).str.strip().str.lower()
                == str(POSITIVE_VALUE).strip().lower()).astype(int)
    if df["_y"].nunique() < 2:
        raise ValueError("Only one class after mapping. Check POSITIVE_VALUE.")
    print(f"YES rate: {df['_y'].mean():.1%}")
else:
    df["_y"] = df[TARGET_COLUMN].astype(float)
    print(f"Target average: {df['_y'].mean():,.1f} {UNIT_LABEL}")

## Section 1: Quick EDA and Data Quality
Boosting is powerful enough to find **any** shortcut in your data, including leaked ones.
So the most important check here is the **leakage sniff test**: an input that predicts the
target almost perfectly by itself usually means it was recorded *after* the outcome.

In [ ]:
# ============================================================
# DATA QUALITY
# ============================================================
m = df[NUM_COLS + CAT_COLS].isna().sum()
print("Missing values:\n", m[m > 0] if m.sum() else "None",
      "\n(Boosting handles blanks in numeric inputs natively; it learns which side of each split they go to.)\n")
print("Duplicate rows:", df.duplicated().sum())
display(df[NUM_COLS].describe().T)

for c in CAT_COLS:
    k = df[c].nunique()
    note = "  ⚠ >255 values: group rare ones into 'Other' first" if k > 255 else (
           "  ⚠ many values: consider grouping rare ones" if k > 50 else "")
    print(f"{c}: {k} distinct values{note}")

# Target picture
if IS_CLF:
    fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(4 * len(CAT_COLS), 3))
    for ax, c in zip(np.atleast_1d(axes), CAT_COLS):
        df.groupby(c)["_y"].mean().sort_values().plot.barh(ax=ax, color="indianred"); ax.set_title(f"YES rate by {c}")
    plt.tight_layout(); plt.show()
else:
    if DATE_COLUMN:
        df.groupby(DATE_COLUMN)["_y"].sum().plot(figsize=(9, 3), title=f"Total {TARGET_COLUMN} over time"); plt.show()
    else:
        df["_y"].hist(bins=40, figsize=(6, 3)); plt.title(TARGET_COLUMN); plt.show()

# ============================================================
# LEAKAGE SNIFF TEST: how well does each numeric input ALONE predict the target?
# ============================================================
print("\nSingle-input strength:")
for c in NUM_COLS:
    s = df[[c, "_y"]].dropna()
    if IS_CLF:
        v = roc_auc_score(s["_y"], s[c]); v = max(v, 1 - v); bad = v > 0.90; lab = "ROC-AUC"
    else:
        v = abs(s[c].corr(s["_y"])); bad = v > 0.95; lab = "|correlation|"
    print(f"  {c:<22} {lab} {v:.2f}" + ("  ⚠ suspiciously strong. Is it known BEFORE the outcome?" if bad else ""))

In [ ]:
# ============================================================
# DATA SIZE CHECK
# Boosting needs more data than linear models to beat them reliably.
# Rough guide: under ~500 rows (or ~200 YES cases), a well-tuned
# linear/logistic model often ties boosting and is far easier to explain.
# ============================================================
n = len(df)
key_n = int(min(df["_y"].sum(), n - df["_y"].sum())) if IS_CLF else n
MODE = ("small" if key_n < (200 if IS_CLF else 1_000) else "large") if DATA_SIZE_MODE == "auto" else DATA_SIZE_MODE
print(f"Rows {n:,}" + (f" | Rare-class cases {key_n:,}" if IS_CLF else "") + f" | Mode: {MODE.upper()}")
if MODE == "small":
    print("""SMALL-DATA PATH
- Slow learning rate, shallow trees, big leaves, extra L2 regularization
- Early stopping OFF (it would hold back data we can't spare); number of rounds is tuned instead
- Repeated CV so results aren't luck of one split
- Always compare against the simple linear benchmark""")

In [ ]:
# ============================================================
# TRAIN/TEST SPLIT
# - With a date: train on the PAST, test on the most RECENT period.
#   A random split would let the model "see the future".
# - Without a date: random split (stratified for classification).
# ============================================================
feature_cols = NUM_COLS + CAT_COLS
USE_TIME = DATE_COLUMN is not None

if USE_TIME:
    cutoff = df[DATE_COLUMN].quantile(1 - TEST_SIZE)
    tr_mask = df[DATE_COLUMN] < cutoff
    X_train, X_test = df.loc[tr_mask, feature_cols], df.loc[~tr_mask, feature_cols]
    y_train, y_test = df.loc[tr_mask, "_y"], df.loc[~tr_mask, "_y"]
    print(f"Time split at {cutoff.date()}: train = before, test = on/after.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        df[feature_cols], df["_y"], test_size=TEST_SIZE, random_state=RANDOM_STATE,
        stratify=df["_y"] if IS_CLF else None)

# ============================================================
# CROSS-VALIDATION SCHEME (used for tuning)
# ============================================================
if USE_TIME:
    CV = TimeSeriesSplit(n_splits=3 if MODE == "small" else 5)   # each fold: train on earlier, validate on later
elif IS_CLF:
    CV = (RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if MODE == "small"
          else StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
else:
    CV = (RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if MODE == "small"
          else KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
print(f"Train {len(X_train):,} | Test {len(X_test):,} | CV: {type(CV).__name__}")

## Section 2: Leakage-Safe Pipeline
- **Numbers pass straight through.** Boosting needs no scaling and handles blanks itself.
- **Categories are turned into codes** and the model is told they're categories, so it groups them sensibly instead of treating "3" as bigger than "2".
- Everything sits **inside a Pipeline**, so during cross-validation the encoding is learned only from each training fold.

In [ ]:
# ============================================================
# PREPROCESSOR (for boosting)
# Unknown categories in future data become -1, which boosting treats as "missing".
# ============================================================
preprocessor = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CAT_COLS),
], verbose_feature_names_out=False)

# Tells the model which output columns are categories (numbers come first, then categories)
CAT_MASK = [False] * len(NUM_COLS) + [True] * len(CAT_COLS)
GB = HistGradientBoostingClassifier if IS_CLF else HistGradientBoostingRegressor

def make_gb(**params):
    """Pipeline = preprocessing + boosting model. Any params override the defaults."""
    settings = {"categorical_features": CAT_MASK, "random_state": RANDOM_STATE}
    settings.update(params)
    return Pipeline([("prep", preprocessor), ("model", GB(**settings))])

# ============================================================
# SIMPLE LINEAR BENCHMARK (needs imputation, scaling, one-hot)
# If boosting can't clearly beat this, use this. It's easier to explain.
# ============================================================
lin_prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_COLS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("oh", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))]), CAT_COLS),
])
linear_model = Pipeline([("prep", lin_prep),
                         ("model", LogisticRegression(max_iter=3000) if IS_CLF else RidgeCV(alphas=np.logspace(-2, 3, 20)))])
print("Pipelines ready.")

## Section 3: Baselines
Three reference points before any tuning:
1. **Dummy:** predicts the average (or "always NO"). The bar for "better than nothing".
2. **Linear/logistic model:** the simple, explainable benchmark. Boosting has to beat it by more than the
   fold-to-fold spread to earn its extra complexity.
3. **Default boosting model:** scikit-learn's out-of-the-box settings. Often already strong.

We show **training** and **validation** scores side by side: a big gap means the model is memorizing.

In [ ]:
# ============================================================
# BASELINE COMPARISON (cross-validated, training data only)
# ============================================================
if IS_CLF:
    SCORING = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision", "F1": "f1"}
    Dummy = DummyClassifier(strategy="prior")
else:
    SCORING = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
    Dummy = DummyRegressor(strategy="mean")
if MAIN_METRIC not in SCORING.values():
    SCORING[MAIN_METRIC] = MAIN_METRIC          # make sure the tuning metric is reported too
KEY = {v: k for k, v in SCORING.items()}[MAIN_METRIC]
SIGN = -1 if MAIN_METRIC.startswith("neg_") else 1   # sklearn stores errors as negatives

def cv_summary(model, name):
    r = cross_validate(model, X_train, y_train, cv=CV, scoring=SCORING, return_train_score=True,
                       n_jobs=-1, error_score=np.nan)
    row = {"model": name}
    for k, s in SCORING.items():
        sg = -1 if s.startswith("neg_") else 1
        row[k] = sg * np.nanmean(r[f"test_{k}"]); row[f"{k} ±"] = np.nanstd(r[f"test_{k}"])
        row[f"{k} (train)"] = sg * np.nanmean(r[f"train_{k}"])
    return row

res = pd.DataFrame([
    cv_summary(Pipeline([("prep", preprocessor), ("model", Dummy)]), "Baseline (dummy)"),
    cv_summary(linear_model, "Linear benchmark"),
    cv_summary(make_gb(), "Default boosting"),
]).set_index("model")
display(res)
print(f"""
HOW TO READ THIS
- Main metric: {KEY}. {'Higher' if SIGN == 1 else 'Lower'} is better.
- ± is the spread across folds. Gaps smaller than ± are effectively ties.
- (train) vs. validation: boosting usually shows a gap; tuning aims to shrink it without losing validation score.""")

## Section 4: Tuning with RandomizedSearchCV (+ Early Stopping)

| Setting | Plain meaning | Overfitting control |
|---|---|---|
| `learning_rate` | How big each correction step is | Smaller = more careful (needs more rounds) |
| `max_iter` | Number of correction rounds (trees) | Too many = learns noise; early stopping picks it for you |
| `max_leaf_nodes` / `max_depth` | How detailed each small tree is | Smaller = simpler patterns |
| `min_samples_leaf` | Minimum rows per final group | Bigger = smoother, safer |
| `l2_regularization` | Penalty on extreme leaf values | Bigger = more cautious |
| `class_weight` (yes/no) | Extra weight on rare YES cases | Helps recall when YES is rare |

**Early stopping** holds back 10% of the training rows *inside each fit* and stops adding rounds when that
holdout stops improving. Great on large data. On small data it wastes rows we can't spare, so the
small-data path turns it **off** and tunes the number of rounds directly.

**Simplicity rule:** among settings within **one standard error** of the best score (the fold spread ÷ √folds),
we pick the one with the **smallest train–validation gap** (least overfit).

In [ ]:
# ============================================================
# SEARCH SPACE
# ============================================================
if MODE == "small":
    base = make_gb(early_stopping=False)
    space = {
        "model__learning_rate": loguniform(0.01, 0.2),
        "model__max_iter": randint(50, 400),              # number of rounds, tuned directly
        "model__max_depth": [2, 3, 4],
        "model__max_leaf_nodes": [4, 8, 15],
        "model__min_samples_leaf": randint(10, 60),
        "model__l2_regularization": loguniform(0.1, 10),
    }
    n_iter = min(N_SEARCH_ITER, 25)
else:
    # max_iter is a ceiling; early stopping decides the actual number of rounds
    base = make_gb(max_iter=1_000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20)
    space = {
        "model__learning_rate": loguniform(0.02, 0.3),
        "model__max_leaf_nodes": randint(8, 64),
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": randint(10, 200),
        "model__l2_regularization": loguniform(1e-3, 10),
    }
    n_iter = N_SEARCH_ITER
if IS_CLF:
    space["model__class_weight"] = [None, "balanced"]

# Large data: tune on a sample (most recent rows for time data), then refit on everything
if SAMPLE_ROWS_FOR_SEARCH and len(X_train) > SAMPLE_ROWS_FOR_SEARCH:
    if USE_TIME:
        Xs, ys = X_train.iloc[-SAMPLE_ROWS_FOR_SEARCH:], y_train.iloc[-SAMPLE_ROWS_FOR_SEARCH:]
    else:
        Xs, _, ys, _ = train_test_split(X_train, y_train, train_size=SAMPLE_ROWS_FOR_SEARCH,
                                        stratify=y_train if IS_CLF else None, random_state=RANDOM_STATE)
    print(f"Tuning on a {len(Xs):,}-row sample.")
else:
    Xs, ys = X_train, y_train

search = RandomizedSearchCV(base, space, n_iter=n_iter, cv=CV, scoring=MAIN_METRIC,
                            random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
search.fit(Xs, ys)

cvr = search.cv_results_
tbl = pd.DataFrame(cvr["params"]).rename(columns=lambda c: c.replace("model__", ""))
tbl["valid"] = SIGN * cvr["mean_test_score"]
tbl["±"] = cvr["std_test_score"]
tbl["train"] = SIGN * cvr["mean_train_score"]
tbl["gap"] = (cvr["mean_train_score"] - cvr["mean_test_score"])   # positive = overfitting
tbl["_i"] = range(len(tbl))
tbl = tbl.sort_values("valid", ascending=(SIGN == -1), kind="stable").reset_index(drop=True)
print(f"Top 10 of {n_iter} combinations ({KEY}):")
display(tbl.drop(columns="_i").head(10))

# ============================================================
# SIMPLICITY RULE: within one standard error of the best, least overfit wins
# (standard error = fold spread ÷ √number of folds)
# ============================================================
best = tbl.iloc[0]
se = best["±"] / np.sqrt(CV.get_n_splits())
within = tbl[(tbl["valid"] >= best["valid"] - se)] if SIGN == 1 else tbl[(tbl["valid"] <= best["valid"] + se)]
chosen = within.loc[within["gap"].idxmin()]
BEST_PARAMS = {k.replace("model__", ""): v for k, v in cvr["params"][int(chosen["_i"])].items()}
final_model = clone(base).set_params(**{f"model__{k}": v for k, v in BEST_PARAMS.items()})

print(f"\nBest {KEY}: {best['valid']:.3f} ± {best['±']:.3f}")
print(f"CHOSEN (least overfit within one standard error, ±{se:.3f}): {KEY} {chosen['valid']:.3f}, gap {chosen['gap']:.3f}")
print({k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in BEST_PARAMS.items()})

if MODE != "small":
    rounds = clone(final_model).fit(X_train, y_train).named_steps["model"].n_iter_
    print(f"Early stopping used {rounds} rounds (out of a 1,000 ceiling).")

In [ ]:
# ============================================================
# TUNED BOOSTING vs. BASELINES (same folds)
# ============================================================
res2 = pd.concat([res, pd.DataFrame([cv_summary(final_model, "Tuned boosting")]).set_index("model")])
display(res2[[KEY, f"{KEY} ±", f"{KEY} (train)"]])

lin, gb = res2.loc["Linear benchmark", KEY], res2.loc["Tuned boosting", KEY]
gain = (gb - lin) * SIGN
if gain <= res2.loc["Tuned boosting", f"{KEY} ±"]:
    print("⚖ Boosting is NOT clearly better than the linear benchmark. The simpler model is the safer choice.")
else:
    print(f"✅ Boosting beats the linear benchmark by {gain:.3f} {KEY}, more than the fold spread.")

## Section 5: Final Test-Set Evaluation

We fit on all training rows and score the locked test set **once**.
- **Yes/no:** ranking quality, then a **threshold** chosen on out-of-fold predictions using your business numbers.
- **Numbers:** typical miss in business units, compared with the baseline and linear benchmark.

In [ ]:
# ============================================================
# TEST SET
# ============================================================
final_model.fit(X_train, y_train)

if IS_CLF:
    proba_test = final_model.predict_proba(X_test)[:, 1]
    print(f"TEST ROC-AUC {roc_auc_score(y_test, proba_test):.3f} | PR-AUC {average_precision_score(y_test, proba_test):.3f} "
          f"(random = {y_test.mean():.3f})")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    RocCurveDisplay.from_predictions(y_test, proba_test, ax=ax[0], name="Boosting"); ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
    PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=ax[1], name="Boosting")
    ax[1].axhline(y_test.mean(), color="k", ls="--", lw=0.8); plt.tight_layout(); plt.show()

    # ---- Out-of-fold probabilities for the threshold (never the test set) ----
    def oof_proba(model, X, y):
        if USE_TIME:   # time data: predict each later chunk from earlier data only
            p = np.full(len(y), np.nan)
            for tr, va in TimeSeriesSplit(n_splits=5).split(X):
                p[va] = clone(model).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[va])[:, 1]
            keep = ~np.isnan(p); return p[keep], y.values[keep]
        cvp = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_predict(model, X, y, cv=cvp, method="predict_proba")[:, 1], y.values

    if BEST_PARAMS.get("class_weight") == "balanced":
        print("Note: class_weight='balanced' was chosen, so probabilities run HIGH (rare YES cases were up-weighted).\n"
              "The ranking is still valid and the threshold below accounts for it; just don't read them as true odds.")
    p_oof, y_oof = oof_proba(final_model, X_train, y_train)
    BASE = len(df)                                   # accounts scored per cycle
    def ev_table(yv, pv):
        rows = []
        for t in np.round(np.arange(0.05, 0.951, 0.025), 3):
            f = pv >= t; nf, tp = int(f.sum()), int((f & (yv == 1)).sum()); sc = BASE / len(yv)
            rows.append({"threshold": t, "flagged": round(nf * sc), "YES caught": round(tp * sc),
                         "precision": tp / nf if nf else np.nan, "recall": tp / max(yv.sum(), 1),
                         "expected value": (tp * VALUE_IF_CAUGHT * ACTION_SUCCESS_RATE - nf * ACTION_COST) * sc})
        return pd.DataFrame(rows)
    tt = ev_table(y_oof, p_oof)
    THRESHOLD = float(tt.loc[tt["expected value"].idxmax(), "threshold"])
    display(tt.iloc[::2].style.format({"precision": "{:.2f}", "recall": "{:.2f}", "flagged": "{:,.0f}",
                                        "YES caught": "{:,.0f}", "expected value": "${:,.0f}"}).hide(axis="index"))
    br = tt.loc[tt["threshold"] == THRESHOLD].iloc[0]
    print(f"Break-even precision: {ACTION_COST / (VALUE_IF_CAUGHT * ACTION_SUCCESS_RATE):.1%}")
    print(f"CHOSEN THRESHOLD {THRESHOLD:.3f}: flag ~{br['flagged']:,.0f} of {BASE:,}, catch {br['recall']:.0%} of YES "
          f"at {br['precision']:.0%} precision → ~${br['expected value']:,.0f} per cycle")
    tt_test = ev_table(y_test.values, proba_test)
    rt = tt_test.loc[tt_test["threshold"] == THRESHOLD].iloc[0]
    print(f"TEST SET at {THRESHOLD:.3f}: precision {rt['precision']:.0%}, recall {rt['recall']:.0%}, ~${rt['expected value']:,.0f} per cycle")
else:
    pred = final_model.predict(X_test)
    lin_pred = clone(linear_model).fit(X_train, y_train).predict(X_test)
    base_pred = np.full(len(y_test), y_train.mean())
    tbl_t = pd.DataFrame({name: [mean_absolute_error(y_test, p), np.sqrt(mean_squared_error(y_test, p)), r2_score(y_test, p)]
                          for name, p in [("Baseline", base_pred), ("Linear", lin_pred), ("Boosting", pred)]},
                         index=["MAE", "RMSE", "R2"])
    display(tbl_t)
    mae = tbl_t.loc["MAE", "Boosting"]
    print(f"Typical miss: {UNIT_LABEL}{mae:,.0f} per row ({mae / y_test.mean():.0%} of the average {TARGET_COLUMN}).")
    print(f"vs. linear benchmark: {UNIT_LABEL}{tbl_t.loc['MAE', 'Linear']:,.0f}; vs. baseline: {UNIT_LABEL}{tbl_t.loc['MAE', 'Baseline']:,.0f}")
    if USE_TIME:
        dates = df.loc[X_test.index, DATE_COLUMN]
        agg = pd.DataFrame({"actual": y_test.values, "forecast": pred, "date": dates.values}).groupby("date").sum()
        agg.plot(figsize=(10, 3.5), title="Test period: total actual vs. forecast"); plt.show()
        total_err = (agg["forecast"].sum() - agg["actual"].sum()) / agg["actual"].sum()
        print(f"Over the whole test period, total forecast is off by {total_err:+.1%} (row-level misses partly cancel out).")
    else:
        plt.figure(figsize=(5, 5)); plt.scatter(y_test, pred, alpha=0.4)
        lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]; plt.plot(lims, lims, "r--")
        plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.show()

## Section 6: What Drives the Predictions?

- **Permutation importance** (test set): which inputs the model relies on most.
- **Partial dependence (PDP) + ICE**: the *shape* of each top driver's effect. Boosting can learn curves,
  thresholds, and plateaus that a linear model would miss.
- **Learning curve**: would more data help?

In [ ]:
# ============================================================
# PERMUTATION IMPORTANCE (test set)
# ============================================================
perm = permutation_importance(final_model, X_test, y_test, scoring=MAIN_METRIC,
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"score drop": perm.importances_mean, "±": perm.importances_std},
                   index=feature_cols).sort_values("score drop", ascending=False)
display(imp)
imp["score drop"][::-1].plot.barh(xerr=imp["±"][::-1], figsize=(7, 0.45 * len(imp) + 1.5), color="steelblue")
plt.title(f"Permutation importance (drop in {KEY} when shuffled)"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# PARTIAL DEPENDENCE + ICE: top 3 numeric drivers
# ============================================================
from sklearn.inspection import partial_dependence

top_num = [c for c in imp.index if c in NUM_COLS][:3]
fig, ax = plt.subplots(1, len(top_num), figsize=(5 * len(top_num), 4))
PartialDependenceDisplay.from_estimator(final_model, X_test, features=top_num, kind="both", subsample=50,
                                        random_state=RANDOM_STATE, ax=np.atleast_1d(ax), grid_resolution=20,
                                        ice_lines_kw={"alpha": 0.15}, pd_line_kw={"color": "red", "lw": 3})
plt.suptitle("Effect of each driver (red = average, grey = individual rows)"); plt.tight_layout(); plt.show()

for c in top_num:
    pdp = partial_dependence(final_model, X_test, [c], grid_resolution=20)
    v, g = pdp["average"][0], pdp["grid_values"][0]
    keep = ~np.isnan(g); v, g = v[keep], g[keep]
    fmt = (lambda x: f"{x:.0%}") if IS_CLF else (lambda x: f"{UNIT_LABEL}{x:,.0f}")
    print(f"- {c}: from {g[0]:,.1f} to {g[-1]:,.1f}, the average prediction moves from {fmt(v[0])} to {fmt(v[-1])} "
          f"(lowest {fmt(v.min())}, highest {fmt(v.max())}).")

In [ ]:
# ============================================================
# LEARNING CURVE: does more data help?
# Still rising on the right = more history would likely help.
# Flat and close together = better INPUTS matter more than more rows.
# ============================================================
if USE_TIME:
    lc_cv = TimeSeriesSplit(n_splits=4)
elif IS_CLF:
    lc_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
else:
    lc_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
sizes, tr_s, va_s = learning_curve(final_model, X_train, y_train, cv=lc_cv, scoring=MAIN_METRIC,
                                   train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1)
plt.figure(figsize=(7, 4))
plt.plot(sizes, SIGN * tr_s.mean(1), "o-", label="Training")
plt.plot(sizes, SIGN * va_s.mean(1), "o-", label="Validation")
plt.xlabel("Training rows"); plt.ylabel(KEY); plt.title("Learning curve"); plt.legend(); plt.show()

## Section 7: Monotonic Constraints (Making the Model Match Business Logic)

Sometimes you **know** the direction of an effect: higher utilization should never *lower* default risk;
a higher price should never *raise* unit sales. With limited data, boosting can learn small wiggles that
break this logic, which confuses stakeholders and can produce odd decisions.

A **monotonic constraint** forces the model's effect to only go up (`+1`) or only go down (`-1`) for that input.
- Usually costs little or no accuracy
- Makes the model safer and easier to defend
- Edit the `MONOTONIC` dictionary below to match your own business logic

In [ ]:
# ============================================================
# MONOTONIC CONSTRAINTS
# +1 = prediction can only rise as the input rises; -1 = only fall.
# Only numeric inputs can be constrained (categories stay 0).
# ============================================================
MONOTONIC = ({"utilization_pct": +1, "avg_days_late_12m": +1} if IS_CLF
             else {"price": -1, "ad_spend": +1})
MONOTONIC = {k: v for k, v in MONOTONIC.items() if k in NUM_COLS}   # ignore names not in your data

cst = [MONOTONIC.get(c, 0) for c in NUM_COLS] + [0] * len(CAT_COLS)
mono_model = clone(final_model).set_params(model__monotonic_cst=cst)
mono_row = cv_summary(mono_model, "Tuned boosting + monotonic")
cmp = pd.concat([res2.loc[["Tuned boosting"]], pd.DataFrame([mono_row]).set_index("model")])
display(cmp[[KEY, f"{KEY} ±"]])
diff = (mono_row[KEY] - res2.loc["Tuned boosting", KEY]) * SIGN
print(f"Constraints applied: {MONOTONIC}")
print("→ Within the spread: keep the constrained model; it's easier to defend." if abs(diff) <= res2.loc["Tuned boosting", f"{KEY} ±"]
      else ("→ Constraints improved the score." if diff > 0 else "→ Constraints cost real accuracy. Double-check the business logic."))

if MONOTONIC:
    mono_model.fit(X_train, y_train)
    feats = list(MONOTONIC)
    fig, ax = plt.subplots(1, len(feats), figsize=(5 * len(feats), 3.5))
    PartialDependenceDisplay.from_estimator(mono_model, X_test, features=feats, ax=np.atleast_1d(ax), grid_resolution=20)
    plt.suptitle("Constrained effects (should only go one way)"); plt.tight_layout(); plt.show()

## Optional: LightGBM or XGBoost

`HistGradientBoosting` is scikit-learn's version of the LightGBM algorithm, so results are usually close.
Reach for the dedicated libraries when data is very large (millions of rows), you need GPU training, or you
want extra options. They drop into the same pipeline:

```python
# from lightgbm import LGBMClassifier, LGBMRegressor        # preinstalled in Colab
# LGBM = LGBMClassifier if IS_CLF else LGBMRegressor
# lgbm = Pipeline([("prep", preprocessor),
#                  ("model", LGBM(n_estimators=2000, learning_rate=0.05, num_leaves=31, min_child_samples=30,
#                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
#                                 reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1))])
# Early stopping in LightGBM: pass a validation set to .fit() with callbacks=[lightgbm.early_stopping(50)]
#
# from xgboost import XGBClassifier, XGBRegressor
# XGB = XGBClassifier if IS_CLF else XGBRegressor
# xgb = Pipeline([("prep", preprocessor),
#                 ("model", XGB(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8,
#                               colsample_bytree=0.8, early_stopping_rounds=50, random_state=RANDOM_STATE))])
```

Same rule as always: switch only if the gain beats the ± spread on cross-validation.

## Section 8: Small Data vs. Large Data

**Small data (under ~1,000 rows or ~200 YES cases):**
- Early stopping **off**; the number of rounds is tuned instead
- Slow learning rate (0.01–0.1), shallow trees (depth 2–4), big leaves, extra L2 penalty
- Repeated CV; always check against the linear benchmark, which often ties
- Use monotonic constraints wherever you know the direction; they act as free regularization

**Large data (tens of thousands of rows and up):**
- Early stopping **on** with a high `max_iter` ceiling; it finds the right number of rounds automatically
- Tune on a sample (`SAMPLE_ROWS_FOR_SEARCH`) and refit on everything
- Bigger trees (`max_leaf_nodes` 31–63) and smaller leaves become safe
- For millions of rows or GPU needs, consider LightGBM / XGBoost
- **Time data:** always validate on later periods (`TimeSeriesSplit`), and add lag features for forecasting

In [ ]:
# ============================================================
# SAVE THE MODEL
# ============================================================
import joblib

FINAL = mono_model if (MONOTONIC and abs(diff) <= res2.loc["Tuned boosting", f"{KEY} ±"]) else final_model
FINAL.fit(pd.concat([X_train, X_test]), pd.concat([y_train, y_test]))   # refit on all labeled data
joblib.dump({"model": FINAL, "features": feature_cols, "task": TASK,
             "threshold": THRESHOLD if IS_CLF else None}, "gradient_boosting_model.joblib")
print("Saved: gradient_boosting_model.joblib",
      "(with monotonic constraints)" if FINAL is mono_model else "")
# Later: b = joblib.load("gradient_boosting_model.joblib")
#        new_df = ...  # same columns; if DATE_COLUMN is set, add the same month / week_of_year columns first
#        preds = b["model"].predict_proba(new_df[b["features"]])[:, 1] if b["task"] == "classification" else b["model"].predict(new_df[b["features"]])

## Final Checklist

- [ ] TASK, target, inputs, and (for forecasting) DATE_COLUMN set correctly
- [ ] Leakage sniff test passed; no input recorded after the outcome
- [ ] Time data validated on later periods, never a random split
- [ ] Boosting beats the dummy **and** the linear benchmark by more than the spread (or I chose linear)
- [ ] Tuned with RandomizedSearchCV; least-overfit setting within one standard error chosen
- [ ] Early stopping on for large data, off for small data
- [ ] Test set used once; results stated in business terms
- [ ] (Yes/no) Threshold from out-of-fold expected value, not 0.5
- [ ] Top drivers and their shapes (PDP) make business sense; monotonic constraints added where logic is known
- [ ] Model saved with feature list and training date